Group 8 
1. Nakabugo Catherine —DSMA/DAY/1522
2. Ssekasanvu Shafick DSMA/DAY/G/1500
3. Maroma John-2025/DSMA/DAY/1986
4. Yawe Rodgers/DSMa/DAY/1535
5. Nakayenga Melisa DSMA/DAY/1558
6. NELSON SSEMPA 2025/DSMA/DAY/0476

    Designing a predictive model that predicts real estate prices basing on features such as year, size and view .

In [3]:
#importing required libraries for data manipulation
import pandas as pd
import numpy as np

In [4]:
#reading the dataset
df= pd.read_csv("real_estate.csv")
df

,price,size,year,view
0,234314.144,643.09,2015,No sea view
1,228581.528,656.22,2009,No sea view
2,281626.336,487.29,2018,Sea view
3,401255.608,1504.75,2015,No sea view
4,458674.256,1275.46,2009,Sea view
...,...,...,...,...
95,252460.400,549.80,2009,Sea view
96,310522.592,1037.44,2009,No sea view
97,383635.568,1504.75,2006,No sea view
98,225145.248,648.29,2015,No sea view


In [5]:
df.isnull().sum()

price    0
size     0
year     0
view     0
dtype: int64

df["view"] = df["view"].map({'Sea view': 1, 'No sea view': 0})
#Machine learning models like linear regression can’t directly handle text labels. They need numeric input.

In [6]:
df['view'] = df['view'].apply(lambda x: 1 if x == "Sea view" else 0) 
#Machine learning models like linear regression can’t directly handle text labels. They need numeric input.

In [7]:
#creating a new variable and dropping 'year'
current_year = 2026
df['property_age'] = current_year - df['year']
df = df.drop('year', axis=1)

In [8]:
#importing necessary libraries for moedling
from sklearn.model_selection import train_test_split #Used to split data into training and testing sets.
from sklearn.preprocessing import LabelEncoder,OneHotEncoder, StandardScaler #Converts labels into numbers. Converts categories into binary columns. Standardizes numerical values.  
from sklearn.compose import ColumnTransformer #Applies different preprocessing to different columns.
from sklearn.linear_model import LinearRegression #Fits a straight-line relationship between features and target.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [9]:
X = df.drop(['price'], axis=1) #the inputs used to make predictions.
y = df['price'] #the value you want the model to predict.

In [10]:
# Identify numeric and categorical columns
categorical_cols = X.select_dtypes(include=['int8', 'category']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numeric features: {numeric_cols}")
print(f"Categorical features: {categorical_cols}")

Numeric features: ['size', 'view', 'property_age']
Categorical features: []


In [11]:
# Create preprocessor(prepares raw data before it is sent to a machine learning model.)
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=123)

In [13]:
# Train model
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [14]:
# Predictions
y_pred = model.predict(X_test)

# Evaluate
print("Coefficients:", model.coef_) 
#These tell you how much the target variable changes when a feature increases by 1 unit, while keeping all other features constant.
print("Intercept:", model.intercept_)
#This is the predicted value when all features are zero.
print("R² Score:", r2_score(y_test, y_pred))
#R² measures how much of the variation in the target is explained by the model.
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
#RMSE measures the average prediction error in the same units as the target variable.

Coefficients: [  219.85544122 55245.70199732 -2691.85890511]
Intercept: 113456.90019688624
R² Score: 0.9137597474307073
RMSE: 21637.616624892373


In [15]:
print(model.feature_names_in_)

['size' 'view' 'property_age']


In [16]:
import joblib
#Joblib allows you to efficiently save and load trained models
joblib.dump(model, "price_model.pkl")

['price_model.pkl']

In [17]:
!pip install streamlit
#Streamlit makes turning Python scripts into interactive web apps fast, simple, and lightweight — without needing deep web development skills.

In [1]:
%%writefile app.py 
#tells Jupyter to Save the code in this cell into a file named app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import joblib

# Load dataset
df = pd.read_csv("real_estate.csv")
df['view'] = df['view'].apply(lambda x: 1 if x == "Sea view" else 0)

#creating a new variable and dropping 'year'
current_year = 2026
df['property_age'] = current_year - df['year']
df = df.drop('year', axis=1)

X = df.drop(['price'], axis=1) 
y = df['price'] 

#Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Load trained model
model = joblib.load("price_model.pkl")
st.title("PRICE Prediction App")
st.write("Enter real estate information below:")

# User inputs
Size = st.number_input("Size (sq ft)", min_value=500, max_value=5000, step=50)
Property_age = st.number_input("property_age")
View = st.selectbox("Sea View?", ["No", "Yes"])
View = 1 if View == "Yes" else 0

# Predict button
if st.button("Predict Price"):
    prediction = model.predict(np.array([[Size,Property_age, View]]))[0]
    st.success(f"Estimated Price: UGX {prediction:,.2f}")

Overwriting app.py


In [ ]:
!streamlit run app.py